# 03. Deep Agent 컨텍스트 엔지니어링 (5가지 컨텍스트 타입)

> 이 노트북은 Deep Agents SDK의 **5가지 컨텍스트 타입**(입력·런타임·압축·격리·장기 메모리) 관점으로 컨텍스트를 엔지니어링해요. 이 5가지 분류는 공식 [Context engineering 문서](https://docs.langchain.com/oss/python/deepagents/context-engineering) 의 Input/Runtime 구분과 압축·오프로딩·격리 전략을 하나의 관점으로 묶어 설명하기 위한 교재 고유 구성이에요. LangChain v1 미들웨어(`@dynamic_prompt`, `@wrap_model_call`) 기반 접근은 **`06_Middleware/03-Context-Engineering.ipynb`** 에서 먼저 익히고 오면 좋아요.

> **왜 컨텍스트 엔지니어링이 중요한가요?**
>
> LLM의 성능은 "무엇을 알고 있느냐"에 달려 있어요. 아무리 뛰어난 모델이라도 **엉뚱한 정보가 컨텍스트에 들어가면 엉뚱한 답**을 해요. 컨텍스트 엔지니어링은 에이전트가 **올바른 정보를, 올바른 시점에, 올바른 양만큼** 갖도록 설계하는 기술이에요.

> 🔑 **비유**: 컨텍스트 엔지니어링은 **요리사의 미장플라스(mise en place)**와 같아요. 요리 시작 전에 필요한 재료만 깔끔하게 준비해두면 효율적으로 요리할 수 있듯이, 에이전트에게 필요한 정보만 정리해서 제공하면 더 정확하고 빠르게 작업을 수행해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Deep Agents의 **5가지 컨텍스트 타입**(입력, 런타임, 압축, 격리, 장기 메모리)의 역할을 설명할 수 있어요
2. `system_prompt`, 메모리 파일(`AGENTS.md`), 스킬(`SKILL.md`), 도구 docstring으로 **입력 컨텍스트**를 구성할 수 있어요
3. `config["context"]` 패턴으로 **런타임 컨텍스트**를 에이전트와 서브에이전트에 전파할 수 있어요
4. 오프로딩·요약을 활용한 **압축 컨텍스트** 전략을 설계할 수 있어요
5. `CompositeBackend(default=StateBackend, routes={"/memories/": StoreBackend})`로 **장기 메모리**를 설정할 수 있어요

## 사전 지식

- 이전 노트북 `02-Deep-Agent-Capabilities.ipynb`: 7가지 하네스 능력 (특히 Filesystem Backend)
- LangGraph 체크포인터(`InMemorySaver`)와 Store API 기본 개념
- `create_deep_agent` 기본 파라미터 (Part 10 노트북 01~02)

## 5가지 컨텍스트 타입 전체 아키텍처

Deep Agents는 에이전트가 "무엇을 알고 있는가"를 5가지 서로 다른 방식으로 제어해요. 컨텍스트 엔지니어링은 에이전트가 올바른 정보를 올바른 시점에 갖도록 설계하는 기술이에요.

```mermaid
flowchart TD
    subgraph INPUT["1. 입력 컨텍스트 (Input Context)"]
        I1["system_prompt"] 
        I2["memory (AGENTS.md)"] 
        I3["skills (Progressive Disclosure)"] 
        I4["tool prompts"]
    end

    subgraph RUNTIME["2. 런타임 컨텍스트 (Runtime Context)"]
        R1["config.get('context', {})"] --> R2["서브에이전트 자동 전파"]
    end

    subgraph COMPRESS["3. 압축 컨텍스트 (Compression Context)"]
        C1["오프로딩 (20K+ 토큰)"] 
        C2["요약 (85% 임계값)"]
    end

    subgraph ISOLATE["4. 격리 컨텍스트 (Isolation Context)"]
        S1["서브에이전트"] --> S2["fresh context"] --> S3["최종 결과만 반환"]
    end

    subgraph LONGTERM["5. 장기 메모리 (Long-term Memory)"]
        L1["CompositeBackend"] --> L2["StateBackend (임시)"]
        L1 --> L3["StoreBackend (/memories/)"]
    end

    INPUT --> AGENT["Deep Agent"] 
    RUNTIME --> AGENT
    COMPRESS --> AGENT
    ISOLATE --> AGENT
    LONGTERM --> AGENT

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef runtime fill:#cce5ff,stroke:#007bff,color:#004085
    classDef compress fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef isolate fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef agent fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class I1,I2,I3,I4 input
    class R1,R2 runtime
    class C1,C2 compress
    class S1,S2,S3 isolate
    class L1,L2,L3 storage
    class AGENT agent
```

| # | 컨텍스트 타입 | 목적 | 핵심 메커니즘 |
|---|--------------|------|---------------|
| 1 | **입력 (Input)** | 에이전트의 기본 지식·역할 설정 | `system_prompt`, `AGENTS.md`, skills |
| 2 | **런타임 (Runtime)** | 실행 시 동적 정보 주입 | `config["context"]` 딕셔너리 |
| 3 | **압축 (Compression)** | 토큰 초과 방지, 긴 작업 지원 | 오프로딩(파일 저장) + 요약(85%) |
| 4 | **격리 (Isolation)** | 서브에이전트 독립 실행 | fresh context, 결과만 반환 |
| 5 | **장기 메모리 (Long-term)** | 세션 간 정보 유지 | `CompositeBackend` + `/memories/` |

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 로드해요
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Deep-Agents-Context"

# ---------------------------------------------------
# Deep Agents 핵심 의존성
# ---------------------------------------------------
from deepagents import create_deep_agent

## 1. 입력 컨텍스트 (Input Context) — 에이전트의 기본 지식

입력 컨텍스트는 에이전트가 시작될 때 갖고 있는 **정적인 지식**이에요. 크게 4가지 소스에서 만들어져요.

```mermaid
flowchart LR
    SP["system_prompt<br>역할, 규칙, 지시"] --> CTX["에이전트의<br>입력 컨텍스트"]
    MEM["memory<br>AGENTS.md 등<br>사용자별 메모리"] --> CTX
    SK["skills<br>SKILL.md<br>Progressive Disclosure"] --> CTX
    TP["tool prompts<br>각 도구의 docstring"] --> CTX

    classDef source fill:#d4edda,stroke:#28a745,color:#155724
    classDef ctx fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class SP,MEM,SK,TP source
    class CTX ctx
```

### 4가지 입력 컨텍스트 소스

| 소스 | 설정 방법 | 특징 |
|------|-----------|------|
| **system_prompt** | `create_deep_agent(system_prompt=...)` | 역할, 행동 규칙, 기본 지시 |
| **memory** | 파일시스템의 `AGENTS.md` | 사용자별/작업별 기억 파일 |
| **skills** | 백엔드의 `SKILL.md` 파일 | 재사용 가능 능력 (Progressive Disclosure) |
| **tool prompts** | 도구 함수의 docstring | 각 도구 사용법 |

> 🔑 **핵심 개념**: Deep Agents에서 `AGENTS.md`는 에이전트가 작업 시작 시 **자동으로 읽는 메모리 파일**이에요. 파일시스템 루트에 이 파일을 두면 에이전트가 이전 작업의 컨텍스트를 이어받을 수 있어요. `SKILL.md`는 프로그레시브 디스클로저(Progressive Disclosure) 방식으로 필요한 시점에만 로드돼요.

> 🎯 **강의 포인트**: `system_prompt`와 `AGENTS.md`의 차이를 강조해주세요. `system_prompt`는 **모든 세션에서 고정**되지만, `AGENTS.md`는 **작업이 진행되면서 에이전트가 직접 업데이트**할 수 있어요. 에이전트가 자신의 메모리를 스스로 관리하는 개념이에요.

In [3]:
# ---------------------------------------------------
# 입력 컨텍스트 구성 실습: system_prompt 설계
# ---------------------------------------------------
# 좋은 system_prompt는 세 가지를 포함해요:
#   1. 역할(Role): 에이전트가 누구인지
#   2. 능력(Capabilities): 무엇을 할 수 있는지
#   3. 행동 규칙(Rules): 어떻게 행동해야 하는지

# ---------------------------------------------------
# [나쁜 예] 너무 짧고 모호한 system_prompt
# ---------------------------------------------------
bad_prompt = "당신은 도움이 되는 에이전트입니다."

# ---------------------------------------------------
# [좋은 예] 역할 + 능력 + 규칙이 명확한 system_prompt
# ---------------------------------------------------
good_prompt = """
당신은 소프트웨어 프로젝트 관리 전문가 에이전트입니다.

## 역할
- 복잡한 개발 작업을 단계별로 분해하고 실행해요
- 파일 생성, 코드 작성, 문서화를 담당해요

## 능력
- 파일시스템을 활용해 작업 결과를 저장해요
- 서브에이전트에게 전문 작업을 위임해요

## 행동 규칙
1. 복잡한 작업은 반드시 write_todos로 계획부터 세워요
2. 중요한 결과는 파일로 저장해요 (메모리 손실 방지)
3. 완료된 작업은 AGENTS.md에 기록해요
"""

# [나쁜 예] system_prompt:
print(f"  길이: {len(bad_prompt)}자")
print(f"  내용: {bad_prompt}")
print()
# [좋은 예] system_prompt:
print(f"  길이: {len(good_prompt)}자")
print(f"  구성: 역할 + 능력 + 행동 규칙")

  길이: 19자
  내용: 당신은 도움이 되는 에이전트입니다.

  길이: 254자
  구성: 역할 + 능력 + 행동 규칙


In [4]:
# ---------------------------------------------------
# AGENTS.md: 에이전트가 자동으로 읽는 메모리 파일
# ---------------------------------------------------
# Deep Agents는 파일시스템 루트에서 AGENTS.md를 찾으면
# 작업 시작 시 자동으로 읽어서 컨텍스트에 포함시켜요
# 에이전트가 스스로 업데이트할 수도 있어요

import tempfile
from pathlib import Path

# ---------------------------------------------------
# 실습용 작업 디렉토리 생성
# ---------------------------------------------------
work_dir = tempfile.mkdtemp(prefix="deep_agents_ctx_")
print(f"작업 디렉토리: {work_dir}")

# ---------------------------------------------------
# AGENTS.md 파일 생성 (에이전트 메모리 파일)
# ---------------------------------------------------
# 이 파일은 에이전트가 시작할 때 자동으로 읽어요
agents_md_path = Path(work_dir) / "AGENTS.md"
agents_md_content = """
# 에이전트 메모리 (AGENTS.md)

## 현재 프로젝트
- 프로젝트명: LangGraph 교재 개발
- 시작일: 2026-01-01
- 담당자: 철

## 완료된 작업
- [x] Part 01 Introduction 완성
- [x] Part 02 LangGraph Basics 완성

## 진행 중인 작업
- [ ] Part 10 Deep Agents 작성 중

## 중요 메모
- 기본 모델: gpt-4o-mini 사용

"""

agents_md_path.write_text(agents_md_content.strip(), encoding="utf-8")
print(f"AGENTS.md 생성 완료: {agents_md_path}")
print()
# 파일 내용:
print(agents_md_content)

작업 디렉토리: /var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/deep_agents_ctx_sxjd7fzi
AGENTS.md 생성 완료: /var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/deep_agents_ctx_sxjd7fzi/AGENTS.md


# 에이전트 메모리 (AGENTS.md)

## 현재 프로젝트
- 프로젝트명: LangGraph 교재 개발
- 시작일: 2026-01-01
- 담당자: 철

## 완료된 작업
- [x] Part 01 Introduction 완성
- [x] Part 02 LangGraph Basics 완성

## 진행 중인 작업
- [ ] Part 10 Deep Agents 작성 중

## 중요 메모
- 기본 모델: gpt-4o-mini 사용




In [5]:
# ---------------------------------------------------
# SKILL.md: Progressive Disclosure 방식의 스킬 파일
# ---------------------------------------------------
# 스킬은 에이전트의 재사용 가능한 전문 지식이에요
# SKILL.md 파일에 스킬을 정의하면, 에이전트가 필요할 때만 로드해요
# (Progressive Disclosure: 필요한 시점에 필요한 내용만)
#
# 공식 로딩 방법 (05-Skills-Memory.ipynb 에서 심화):
#   agent = create_deep_agent(
#       model="openai:gpt-4o-mini",
#       skills=[str(Path(work_dir) / "skills")],  # SKILL.md가 들어있는 디렉토리 목록
#       backend=FilesystemBackend(root_dir=work_dir, virtual_mode=True),
#   )

# 스킬 디렉토리 생성
skills_dir = Path(work_dir) / "skills" / "code_review"
skills_dir.mkdir(parents=True, exist_ok=True)

# SKILL.md 작성 예시 (프론트매터 필드: name, description 필수)
skill_md_path = skills_dir / "SKILL.md"
skill_md_content = """
---
name: code-review
description: >
  Python 코드를 리뷰하고 개선점을 제안하는 스킬이에요.
  보안, 성능, 가독성 관점에서 분석해요.
metadata:
  version: "1.0.0"
  category: "development"
---

# Code Review Skill

## 리뷰 체크리스트

### 보안
- SQL 인젝션 취약점 확인
- 하드코딩된 비밀번호/API 키 검사
- 입력값 검증 여부

### 성능
- N+1 쿼리 패턴 확인
- 불필요한 루프 중첩
- 메모리 누수 가능성

### 가독성
- 함수명, 변수명 명확성
- docstring 포함 여부
- 복잡도 임계값 (사이클로매틱 복잡도 < 10)
"""

skill_md_path.write_text(skill_md_content.strip(), encoding="utf-8")
print(f"SKILL.md 생성 완료: {skill_md_path}")
print()
# [개념] Progressive Disclosure (공식 스킬 문서):
#   - Match : description/frontmatter만 시스템 프롬프트로 읽혀요 (~100 토큰)
#   - Read  : 에이전트가 관련 있다고 판단하면 SKILL.md 본문 전체를 read_file
#   - Execute: 스킬 내부 지시에 따라 작업 실행
print()
# [장점] 스킬이 수십 개여도 매치된 소수만 본문을 로드해 토큰을 절약해요


SKILL.md 생성 완료: /var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/deep_agents_ctx_sxjd7fzi/skills/code_review/SKILL.md




In [6]:
# ---------------------------------------------------
# 입력 컨텍스트를 완성한 에이전트 생성
# ---------------------------------------------------
# 4가지 입력 컨텍스트 소스를 모두 활용해요

from langchain.tools import tool


@tool
def search_docs(query: str) -> str:
    """프로젝트 문서에서 관련 내용을 검색해요.

    이 도구는 프로젝트 문서베이스를 검색해서 관련된 내용을 반환해요.
    정확한 키워드나 개념 설명을 쿼리로 사용하면 좋아요.

    Args:
        query: 검색할 키워드 또는 질문

    Returns:
        관련 문서 내용 (없으면 빈 문자열)
    """
    # 데모용: 실제로는 벡터 검색 등을 사용해요
    mock_docs = {
        "langraph": "LangGraph는 상태 기반 AI 워크플로 프레임워크예요.",
        "deep agent": "Deep Agents는 복잡한 작업을 자율적으로 수행하는 에이전트예요.",
    }
    for key, value in mock_docs.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 문서를 찾지 못했어요."


# 입력 컨텍스트 구성 요약:
# ==================================================
# 1. system_prompt: 역할 + 능력 + 행동 규칙
# 2. memory (AGENTS.md): 프로젝트 현황, 완료 작업
# 3. skills (SKILL.md): code-review 스킬 정의
# 4. tool prompts: search_docs docstring
print()

from deepagents.backends import FilesystemBackend

# 4가지 입력 컨텍스트 소스 모두 활용
# virtual_mode=False: 실제 파일 시스템 경로 사용 (기본값 변경 예고)
input_ctx_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[search_docs],              # 도구 프롬프트 = 도구의 docstring
    system_prompt=good_prompt,        # system_prompt: 역할+능력+규칙
    backend=FilesystemBackend(        # AGENTS.md, SKILL.md 자동 로드
        root_dir=work_dir,
        virtual_mode=False,
    ),
)
# 입력 컨텍스트 에이전트 생성 완료

## 2. 런타임 컨텍스트 (Runtime Context) — 실행 시 동적 주입

런타임 컨텍스트는 에이전트가 **실행될 때마다** 동적으로 주입하는 정보예요. 사용자별, 세션별, 시간별로 다른 정보를 에이전트에게 전달할 수 있어요.

```mermaid
flowchart LR
    USER["사용자 A<br>user_id: 'alice'"] --> CONFIG["config dict<br>{\"context\": {\"user_id\": \"alice\", \"lang\": \"ko\"}}"] 
    CONFIG --> MAIN["메인 에이전트"]
    MAIN -->|"자동 전파"| SUB1["서브에이전트 1"]
    MAIN -->|"자동 전파"| SUB2["서브에이전트 2"]
    MAIN -->|"자동 전파"| SUB3["서브에이전트 3"]

    classDef user fill:#d4edda,stroke:#28a745,color:#155724
    classDef config fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef agent fill:#cce5ff,stroke:#007bff,color:#004085
    classDef sub fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class USER user
    class CONFIG config
    class MAIN agent
    class SUB1,SUB2,SUB3 sub
```

> 🔑 **핵심 개념**: 런타임 컨텍스트는 `config` 딕셔너리의 `"context"` 키에 담아서 전달해요. 메인 에이전트가 서브에이전트를 실행할 때 이 값이 **자동으로 전파**되기 때문에, 사용자 정보를 한 번만 설정하면 전체 에이전트 체인에 전달돼요.

> 💡 **실무 팁**: `user_id`, `language`, `permissions`, `session_id` 등 사용자별로 다른 값을 런타임 컨텍스트에 담으면 멀티테넌트 에이전트를 쉽게 구현할 수 있어요. 같은 에이전트 코드로 다른 사용자에게 맞춤형 서비스를 제공해요.

In [7]:
# ---------------------------------------------------
# 런타임 컨텍스트: config["context"] 패턴
# ---------------------------------------------------
# 에이전트가 노드 함수 내에서 config로 런타임 정보를 읽는 방법이에요

from langchain_core.runnables import RunnableConfig

# ---------------------------------------------------
# 런타임 컨텍스트를 읽는 노드 함수 예시
# ---------------------------------------------------
def my_agent_node(state: dict, config: RunnableConfig) -> dict:
    """런타임 컨텍스트를 읽는 노드 함수 예시예요."""
    # config.get("context", {})로 런타임 컨텍스트를 읽어요
    ctx = config.get("context", {}) if config else {}

    # 사용자별 맞춤 동작
    user_id = ctx.get("user_id", "anonymous")    # 기본값: anonymous
    language = ctx.get("language", "ko")          # 기본값: 한국어
    permissions = ctx.get("permissions", [])       # 기본값: 빈 목록

    print(f"  [노드 실행] 사용자: {user_id}")
    print(f"  [노드 실행] 언어: {language}")
    print(f"  [노드 실행] 권한: {permissions}")

    # 권한에 따라 다른 동작
    if "admin" in permissions:
        response = f"[{user_id}] 관리자 전용 응답: 모든 데이터에 접근 가능해요"
    else:
        response = f"[{user_id}] 일반 사용자 응답: 공개 데이터만 접근 가능해요"

    return {"response": response}


# ---------------------------------------------------
# 사용자별 다른 런타임 컨텍스트로 실행
# ---------------------------------------------------
# 런타임 컨텍스트 시뮬레이션:
# --------------------------------------------------

# 사용자 A (일반 사용자)
# [사용자 A]
config_user_a: RunnableConfig = {
    "configurable": {
        "thread_id": "thread-alice",
    },
    "context": {              # 런타임 컨텍스트 주입
        "user_id": "alice",
        "language": "ko",
        "permissions": ["read"],
    },
}
my_agent_node({}, config_user_a)

print()

# 사용자 B (관리자)
# [사용자 B (관리자)]
config_user_b: RunnableConfig = {
    "configurable": {
        "thread_id": "thread-bob",
    },
    "context": {              # 런타임 컨텍스트 주입
        "user_id": "bob",
        "language": "en",
        "permissions": ["read", "write", "admin"],
    },
}
my_agent_node({}, config_user_b)

  [노드 실행] 사용자: alice
  [노드 실행] 언어: ko
  [노드 실행] 권한: ['read']

  [노드 실행] 사용자: bob
  [노드 실행] 언어: en
  [노드 실행] 권한: ['read', 'write', 'admin']


{'response': '[bob] 관리자 전용 응답: 모든 데이터에 접근 가능해요'}

### 런타임 컨텍스트 자동 전파

```python
# 1단계: 메인 에이전트 실행 (config에 context 주입)
result = main_agent.invoke(
    {"messages": [HumanMessage(content="작업 시작")]},
    config={
        "configurable": {"thread_id": "session-001"},
        "context": {                  # 이 값이 자동 전파돼요
            "user_id": "alice",
            "language": "ko",
            "project": "langraph-tutorial",
        }
    }
)

# 2단계: 메인 에이전트가 서브에이전트 호출 시
# -> context 딕셔너리가 자동으로 서브에이전트에게 전달돼요
# -> 서브에이전트도 config.get("context") 로 같은 값을 읽어요
# -> 개발자가 별도로 전달 코드를 작성하지 않아도 돼요!
```

멀티에이전트 시스템에서 사용자 정보를 한 번만 설정해도 모든 서브에이전트가 동일한 context를 공유해요.

**활용 사례:**
- `user_id`: 사용자별 데이터 격리
- `language`: 다국어 응답 생성
- `permissions`: 권한에 따른 동작 제어
- `session_id`: 세션 추적 및 로깅
- `api_keys`: 사용자별 외부 서비스 인증


## 3. 압축 컨텍스트 (Compression Context) — 무한 작업 지원

Deep Agents는 긴 작업을 수행하다 보면 대화 히스토리가 컨텍스트 윈도우를 넘어설 수 있어요. 이를 해결하는 두 가지 압축 전략을 살펴볼게요.

```mermaid
flowchart TD
    TOKENS["대화 히스토리 증가"] --> CHECK{"20,000 토큰<br>초과?"}
    CHECK -->|"아니오"| KEEP["그대로 유지"]
    CHECK -->|"예"| STRATEGY{"압축 전략 선택"}

    STRATEGY -->|"대용량 데이터"| OFFLOAD["오프로딩 (Offloading)<br>파일시스템에 저장<br>→ 경로 참조로 대체"]
    STRATEGY -->|"대화 히스토리"| SUMMARIZE["요약 (Summarization)<br>85% 임계값<br>→ LLM 요약으로 압축"]

    OFFLOAD --> PATH["'/workspace/data.json'<br>파일로 저장"]
    SUMMARIZE --> SUMMARY["구조화된 요약<br>+ 최근 15% 원본 유지"]

    PATH --> CONTINUE["작업 계속"]
    SUMMARY --> CONTINUE
    KEEP --> CONTINUE

    classDef check fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef strategy fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef compress fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef normal fill:#d4edda,stroke:#28a745,color:#155724

    class CHECK check
    class STRATEGY strategy
    class OFFLOAD,SUMMARIZE,PATH,SUMMARY compress
    class TOKENS,KEEP,CONTINUE normal
```

### 두 가지 압축 전략

| 전략 | 트리거 | 동작 | 원본 보존 |
|------|--------|------|----------|
| **오프로딩** | 컨텍스트에 20K+ 토큰 데이터 | 파일시스템에 저장 → 경로로 참조 | 파일시스템에 완전 보존 |
| **요약** | 전체 히스토리가 85% 임계값 초과 | LLM으로 요약 생성 + 최근 15% 원본 | 요약 + 최근 메시지만 유지 |

> 🔑 **핵심 개념**: 오프로딩과 요약은 **상호 보완적**이에요. 대용량 데이터(JSON, CSV, 코드 등)는 오프로딩으로 파일에 저장하고 경로만 컨텍스트에 남겨요. 오랜 대화 히스토리는 요약으로 핵심 정보만 남기고 압축해요.

> ⚠️ **자주 하는 실수**: 요약 압축 후에는 에이전트가 초반의 세부 정보를 정확히 기억하지 못할 수 있어요. **반드시 기억해야 하는 값**(파일 경로, 중간 결과, 설정값 등)은 파일시스템이나 Store에 명시적으로 저장하도록 system_prompt에 지시해야 해요.

In [8]:
# ---------------------------------------------------
# 오프로딩 (Offloading) 패턴 실습
# ---------------------------------------------------
# 대용량 데이터를 컨텍스트에 직접 담지 않고
# 파일로 저장한 뒤 경로만 참조하는 패턴이에요

import json
from pathlib import Path

# ---------------------------------------------------
# [나쁜 예] 대용량 데이터를 컨텍스트에 직접 담기
# ---------------------------------------------------
# 수천 개의 레코드를 메시지에 직접 포함하면 토큰이 폭발해요
large_dataset = [
    {"id": i, "name": f"사용자_{i}", "score": i * 10}
    for i in range(100)  # 실제로는 수천~수만 건이에요
]

bad_message = f"다음 데이터를 분석해줘: {json.dumps(large_dataset, ensure_ascii=False)}"
print(f"[나쁜 예] 메시지 크기: {len(bad_message)}자 → 토큰 낭비!")
print()

# ---------------------------------------------------
# [좋은 예] 오프로딩 패턴: 파일에 저장 후 경로 참조
# ---------------------------------------------------
# 1단계: 대용량 데이터를 파일에 저장
data_file_path = Path(work_dir) / "analysis_data.json"
data_file_path.write_text(
    json.dumps(large_dataset, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# 2단계: 경로만 메시지에 포함
good_message = f"/workspace/analysis_data.json 파일의 데이터를 분석해줘"
print(f"[좋은 예] 메시지 크기: {len(good_message)}자 → 경로만 참조")
print(f"  파일 크기: {data_file_path.stat().st_size:,}바이트")
print(f"  메시지: {good_message}")
print()
# [원리] 에이전트가 파일을 read 도구로 읽어서 분석해요
#        컨텍스트 윈도우에는 파일 경로(~50자)만 들어가요
print(f"       절약 토큰: ~{(len(bad_message) - len(good_message)) // 4:,}개")

[나쁜 예] 메시지 크기: 4383자 → 토큰 낭비!

[좋은 예] 메시지 크기: 43자 → 경로만 참조
  파일 크기: 6,771바이트
  메시지: /workspace/analysis_data.json 파일의 데이터를 분석해줘

       절약 토큰: ~1,085개


### 요약 압축(Summarization) 동작 흐름

**설정값:**
- 압축 임계값: 약 20,000 토큰 (전체 히스토리)
- 요약 비율: 85% (오래된 메시지의 85%를 요약으로 대체)
- 유지 비율: 15% (최근 메시지는 원본 그대로 유지)

**압축 전 (총 25,000 토큰):**

| 메시지 | 토큰 |
|--------|------|
| 사용자: 파일 A를 분석해줘 | 500 |
| 에이전트: 파일 A 분석 완료 | 1,200 |
| 사용자: 파일 B도 분석해줘 | 500 |
| 에이전트: 파일 B 분석 완료 | 1,200 |
| ... (중간 대화 20개 생략) | 18,000 |
| 최근 메시지 3개 | 3,600 |

**압축 후 (~6,600 토큰, 74% 감소):**

| 메시지 | 토큰 |
|--------|------|
| [요약] 이전 대화: 파일 A와 B 분석, 주요 결과 요약 | ~3,000 |
| [원본] 최근: 두 파일을 비교해줘 | 500 |
| [원본] 최근: 비교 작업 진행 중 | 1,000 |
| [원본] 최근: 비교 결과 반환 | 2,100 |


In [9]:
# ---------------------------------------------------
# 압축 컨텍스트 전략: system_prompt로 안내하기
# ---------------------------------------------------
# 압축이 일어나도 중요한 정보가 손실되지 않도록
# system_prompt에 명시적 지시를 포함하는 게 좋아요

# ---------------------------------------------------
# 압축 내성(Compression-Resilient) system_prompt 예시
# ---------------------------------------------------
compression_resilient_prompt = """
당신은 장기 프로젝트를 수행하는 에이전트입니다.

## 중요 데이터 보존 규칙

다음 정보는 반드시 파일에 저장하세요 (메모리 압축 후에도 유지됨):

1. **중간 결과**: 각 단계의 결과를 `/workspace/results/step_N.json`에 저장
2. **진행 상황**: `AGENTS.md`에 완료된 작업 목록 업데이트
3. **중요 수치**: 분석 결과의 핵심 수치는 `/workspace/summary.json`에 기록
4. **파일 경로**: 생성한 파일들의 경로 목록을 `/workspace/file_index.txt`에 유지

컨텍스트 압축이 발생해도 이 파일들은 파일시스템에 안전하게 보존됩니다.
"""

# [압축 내성 system_prompt 전략]
# --------------------------------------------------
print(compression_resilient_prompt)
print()
# [핵심] 파일시스템은 컨텍스트 압축에 영향받지 않아요
#        중요한 정보는 항상 파일로 저장 = 안전한 장기 메모리


당신은 장기 프로젝트를 수행하는 에이전트입니다.

## 중요 데이터 보존 규칙

다음 정보는 반드시 파일에 저장하세요 (메모리 압축 후에도 유지됨):

1. **중간 결과**: 각 단계의 결과를 `/workspace/results/step_N.json`에 저장
2. **진행 상황**: `AGENTS.md`에 완료된 작업 목록 업데이트
3. **중요 수치**: 분석 결과의 핵심 수치는 `/workspace/summary.json`에 기록
4. **파일 경로**: 생성한 파일들의 경로 목록을 `/workspace/file_index.txt`에 유지

컨텍스트 압축이 발생해도 이 파일들은 파일시스템에 안전하게 보존됩니다.




## 4. 격리 컨텍스트 (Isolation Context) — 서브에이전트 독립 실행

서브에이전트는 메인 에이전트와 **완전히 독립된 컨텍스트**에서 실행돼요. 각 서브에이전트는 자신의 작업에만 집중하고, 완료 후에는 결과만 메인 에이전트에게 반환해요.

```mermaid
flowchart LR
    MAIN["메인 에이전트<br>(긴 대화 히스토리 보유)"] 
    
    subgraph ISO1["서브에이전트 1 (격리 컨텍스트)"]
        T1["task 지시사항"] --> W1["독립적 실행"]
        W1 --> R1["결과만 반환"]
    end
    
    subgraph ISO2["서브에이전트 2 (격리 컨텍스트)"]
        T2["task 지시사항"] --> W2["독립적 실행"]
        W2 --> R2["결과만 반환"]
    end

    MAIN -->|"task 도구: 지시사항만 전달"| ISO1
    MAIN -->|"task 도구: 지시사항만 전달"| ISO2
    R1 -->|"결과만"| MAIN
    R2 -->|"결과만"| MAIN

    classDef main fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef iso fill:#cce5ff,stroke:#007bff,color:#004085
    classDef result fill:#d4edda,stroke:#28a745,color:#155724

    class MAIN main
    class T1,W1,T2,W2 iso
    class R1,R2 result
```

> 🔑 **핵심 개념**: 격리 컨텍스트의 핵심은 **컨텍스트 오염 방지**예요. 메인 에이전트의 수천 토큰 대화 히스토리가 서브에이전트에게 전달되지 않아요. 서브에이전트는 지시사항만 받고, 자신의 짧은 컨텍스트에서 집중해서 작업해요.

> 💡 **실무 팁**: 격리 컨텍스트는 **비용 절감**에도 효과적이에요. 메인 에이전트의 긴 히스토리 없이 짧은 지시사항과 결과만 주고받으니까요. 복잡한 병렬 작업은 서브에이전트에게 위임하면 토큰 효율이 크게 향상돼요.

In [10]:
# ---------------------------------------------------
# 격리 컨텍스트 이해: 서브에이전트가 받는 것과 반환하는 것
# ---------------------------------------------------

# 격리 컨텍스트 동작 방식:
# ==================================================
print()

# 메인 에이전트의 상황
main_agent_history = [
    "HumanMessage: 'LangGraph 교재를 작성해줘'",
    "AIMessage: '네, 먼저 커리큘럼을 설계할게요'",
    "ToolMessage (write_todos): '1. 커리큘럼 설계...',",
    "AIMessage: 'Part 1부터 시작할게요'",
    "... (수천 토큰의 대화 히스토리)",
    "AIMessage: 'Part 10을 위해 서브에이전트에게 요청할게요'",
]

# [메인 에이전트 컨텍스트]
print(f"  히스토리: {len(main_agent_history)}개 메시지 (약 5,000+ 토큰)")
for msg in main_agent_history:
    print(f"  - {msg}")
print()

# 서브에이전트가 받는 것
subagent_receives = {
    "task": "Deep Agents 컨텍스트 엔지니어링 섹션의 코드 예제를 작성해줘",
    "context": {"user_id": "alice", "language": "ko"},  # 런타임 컨텍스트 자동 전파
    # 메인 에이전트의 긴 히스토리는 전달되지 않아요!
}

# [서브에이전트가 받는 것] (격리 컨텍스트)
print(f"  task: '{subagent_receives['task']}'")
print(f"  context: {subagent_receives['context']}  ← 런타임 ctx 전파")
print(f"  히스토리: 없음! (fresh start)")
print()

# 서브에이전트가 반환하는 것
subagent_returns = """```python
# 컨텍스트 엔지니어링 예제
ctx = config.get("context", {})
user_id = ctx.get("user_id", "anonymous")
```"""

# [서브에이전트가 반환하는 것]
print(f"  결과: 작성된 코드 예제")
print(f"  (서브에이전트의 내부 실행 과정은 반환되지 않아요)")
print()
# [결론] 메인 에이전트의 컨텍스트 오염 없이 깨끗한 결과만 받아요


  히스토리: 6개 메시지 (약 5,000+ 토큰)
  - HumanMessage: 'LangGraph 교재를 작성해줘'
  - AIMessage: '네, 먼저 커리큘럼을 설계할게요'
  - ToolMessage (write_todos): '1. 커리큘럼 설계...',
  - AIMessage: 'Part 1부터 시작할게요'
  - ... (수천 토큰의 대화 히스토리)
  - AIMessage: 'Part 10을 위해 서브에이전트에게 요청할게요'

  task: 'Deep Agents 컨텍스트 엔지니어링 섹션의 코드 예제를 작성해줘'
  context: {'user_id': 'alice', 'language': 'ko'}  ← 런타임 ctx 전파
  히스토리: 없음! (fresh start)

  결과: 작성된 코드 예제
  (서브에이전트의 내부 실행 과정은 반환되지 않아요)



In [11]:
# ---------------------------------------------------
# 격리 컨텍스트 실습: 서브에이전트 정의 및 활용
# ---------------------------------------------------

# 서브에이전트 딕셔너리로 격리 실행 설정
# system_prompt와 tools는 필수 키예요
code_writer_subagent = {
    "name": "code_writer",
    "description": (
        "Python 코드를 작성하는 전문가입니다. "
        "명확하고 한글 주석이 포함된 교육용 코드를 작성해요. "
        "특정 개념에 대한 코드 예제가 필요할 때 사용하세요."
    ),
    "model": "openai:gpt-4o-mini",
    "system_prompt": (
        "당신은 Python 코드 작성 전문가입니다. "
        "한글 주석이 포함된 명확하고 교육적인 코드를 작성해주세요."
    ),
    "tools": [],
    # 서브에이전트는 메인의 히스토리 없이 독립 실행돼요
    # 지시사항만 전달받아 최소한의 토큰으로 결과를 반환해요
}

reviewer_subagent = {
    "name": "reviewer",
    "description": (
        "코드를 리뷰하고 개선점을 찾는 전문가입니다. "
        "보안, 성능, 가독성 관점에서 코드를 검토해요. "
        "작성된 코드의 품질을 확인할 때 사용하세요."
    ),
    "model": "openai:gpt-4o-mini",
    "system_prompt": (
        "당신은 코드 리뷰 전문가입니다. "
        "보안, 성능, 가독성 관점에서 코드를 검토하고 개선점을 제시해주세요."
    ),
    "tools": [],
}

# 격리 컨텍스트 서브에이전트 구성:
# --------------------------------------------------
for sa in [code_writer_subagent, reviewer_subagent]:
    print(f"서브에이전트: {sa['name']}")
    print(f"  설명: {sa['description'][:70]}...")
    print(f"  모델: {sa['model']}")
    print(f"  격리: 메인 에이전트 히스토리 없이 독립 실행")
    print()

isolated_main = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],
    system_prompt=(
        "당신은 코드 작성 프로젝트의 오케스트레이터입니다. "
        "코드 작성은 code_writer에게, 리뷰는 reviewer에게 위임하세요."
    ),
    subagents=[code_writer_subagent, reviewer_subagent],
)
# 격리 컨텍스트 오케스트레이터 에이전트 생성 완료

서브에이전트: code_writer
  설명: Python 코드를 작성하는 전문가입니다. 명확하고 한글 주석이 포함된 교육용 코드를 작성해요. 특정 개념에 대한 코드 예제가...
  모델: openai:gpt-4o-mini
  격리: 메인 에이전트 히스토리 없이 독립 실행

서브에이전트: reviewer
  설명: 코드를 리뷰하고 개선점을 찾는 전문가입니다. 보안, 성능, 가독성 관점에서 코드를 검토해요. 작성된 코드의 품질을 확인할 때 ...
  모델: openai:gpt-4o-mini
  격리: 메인 에이전트 히스토리 없이 독립 실행



## 5. 장기 메모리 (Long-term Memory) — 세션 간 정보 유지

Deep Agents의 장기 메모리는 `CompositeBackend`를 사용해서 구현해요. 일반 파일 접근은 `StateBackend`(임시)로, `/memories/` 경로에 대한 접근은 `StoreBackend`(영구)로 라우팅해요.

```mermaid
flowchart LR
    AGENT["Deep Agent"] --> COMPOSITE["CompositeBackend\
(경로별 라우팅)"]
    
    COMPOSITE -->|"일반 경로<br>/workspace/..."| STATE["StateBackend<br>임시 저장<br>(세션 내)"]
    COMPOSITE -->|"/memories/ 경로"| STORE["StoreBackend<br>영구 저장<br>(세션 간 유지)"]

    STATE -->|"세션 종료 시 삭제"| LOST["사라짐"]
    STORE -->|"세션 재시작 후에도 유지"| PERSIST["보존됨"]

    classDef agent fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef composite fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef temp fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef persist fill:#d4edda,stroke:#28a745,color:#155724
    classDef result fill:#cce5ff,stroke:#007bff,color:#004085

    class AGENT agent
    class COMPOSITE composite
    class STATE temp
    class STORE persist
    class LOST result
    class PERSIST persist
```

### CompositeBackend 라우팅 규칙

| 경로 패턴 | 백엔드 | 데이터 수명 |
|----------|--------|------------|
| `/workspace/*`, `/tmp/*` (일반) | `StateBackend` | 세션 내 임시 |
| `/memories/*` | `StoreBackend` | 세션 간 영구 |

> 🔑 **핵심 개념**: 에이전트가 `/memories/user_alice.md` 파일에 정보를 쓰면 StoreBackend가 저장해요. 다음 세션에서 에이전트가 같은 경로를 읽으면 이전 세션의 정보를 그대로 읽어요. 에이전트 입장에서는 그냥 파일을 읽고 쓰는 것처럼 보이지만, 내부적으로는 영구 저장소를 사용하고 있어요.

> 💡 **실무 팁**: `/memories/` 경로 규칙을 system_prompt에 명시하면 에이전트가 스스로 중요한 정보를 장기 메모리에 저장해요. "사용자 선호도는 `/memories/user_{user_id}.md`에 저장하세요"라고 지시하면 돼요.

In [12]:
# ---------------------------------------------------
# LangGraph Store: 장기 메모리의 기반 스토리지
# ---------------------------------------------------
# StoreBackend는 LangGraph의 Store API를 파일시스템처럼 래핑한 것이에요
# Namespace와 Key로 데이터를 구조적으로 저장해요

# InMemoryStore: 개발/테스트용 인메모리 Store
# 실제 프로덕션에서는 PostgresStore, Redis 등을 사용해요
from langgraph.store.memory import InMemoryStore

# ---------------------------------------------------
# Store 생성 및 기본 사용법
# ---------------------------------------------------
store = InMemoryStore()

# Store에 데이터 저장: (namespace, key, value)
# namespace: 데이터 카테고리 (사용자ID, 프로젝트 등으로 격리)
# key: 고유 식별자
# value: 저장할 딕셔너리

# 사용자 A의 선호도 저장
store.put(
    namespace=("memories", "user_alice"),  # 네임스페이스: 사용자별 격리
    key="preferences",                     # 키: 선호도
    value={
        "language": "ko",
        "model": "gpt-4o-mini",
        "response_style": "detailed",
        "last_project": "LangGraph 교재",
    }
)

# 사용자 A의 작업 기록 저장
store.put(
    namespace=("memories", "user_alice"),
    key="task_history",
    value={
        "completed": ["Part01", "Part02", "Part03"],
        "in_progress": "Part10",
        "total_sessions": 12,
    }
)

# Store에 데이터 저장 완료
print()

# ---------------------------------------------------
# Store에서 데이터 읽기
# ---------------------------------------------------
prefs = store.get(namespace=("memories", "user_alice"), key="preferences")
history = store.get(namespace=("memories", "user_alice"), key="task_history")

# [사용자 alice 장기 메모리 조회]
if prefs:
    print(f"  선호도: {prefs.value}")
if history:
    print(f"  작업 기록: {history.value}")


  선호도: {'language': 'ko', 'model': 'gpt-4o-mini', 'response_style': 'detailed', 'last_project': 'LangGraph 교재'}
  작업 기록: {'completed': ['Part01', 'Part02', 'Part03'], 'in_progress': 'Part10', 'total_sessions': 12}


In [13]:
# ---------------------------------------------------
# CompositeBackend 구성: StateBackend + StoreBackend
# ---------------------------------------------------
# 경로별로 다른 백엔드를 라우팅하는 핵심 패턴이에요
# /memories/ 경로 → StoreBackend (영구 저장)
# 그 외 경로     → StateBackend (임시 저장)

# CompositeBackend 설정 코드:
# --------------------------------------------------
composite_code = '''
from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.memory import InMemoryStore

# LangGraph Store 생성 (실제로는 PostgresStore 사용 권장)
store = InMemoryStore()

# CompositeBackend: 경로별 라우팅
composite_backend = CompositeBackend(
    default=StateBackend(),  # 그 외 모든 경로는 StateBackend (임시 저장)
    routes={
        # /memories/ 경로는 StoreBackend (세션 간 영구 저장)
        "/memories/": StoreBackend(store=store),
    },
)
'''
print(composite_code)

print()
# 라우팅 예시:
routing_examples = [
    ("/workspace/notes.md",          "StateBackend",  "세션 종료 후 삭제"),
    ("/tmp/analysis.json",           "StateBackend",  "세션 종료 후 삭제"),
    ("/memories/user_alice.md",      "StoreBackend",  "세션 간 영구 유지"),
    ("/memories/project_summary.md", "StoreBackend",  "세션 간 영구 유지"),
]

for path, backend, lifecycle in routing_examples:
    print(f"  {path:42s} → {backend:14s} ({lifecycle})")



from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.memory import InMemoryStore

# LangGraph Store 생성 (실제로는 PostgresStore 사용 권장)
store = InMemoryStore()

# CompositeBackend: 경로별 라우팅
composite_backend = CompositeBackend(
    default=StateBackend(),  # 그 외 모든 경로는 StateBackend (임시 저장)
    routes={
        # /memories/ 경로는 StoreBackend (세션 간 영구 저장)
        "/memories/": StoreBackend(store=store),
    },
)


  /workspace/notes.md                        → StateBackend   (세션 종료 후 삭제)
  /tmp/analysis.json                         → StateBackend   (세션 종료 후 삭제)
  /memories/user_alice.md                    → StoreBackend   (세션 간 영구 유지)
  /memories/project_summary.md               → StoreBackend   (세션 간 영구 유지)


In [14]:
# ---------------------------------------------------
# 장기 메모리를 활용하는 에이전트 생성
# ---------------------------------------------------
# system_prompt에 /memories/ 경로 사용 규칙을 명시해요

longterm_memory_prompt = """
당신은 개인 비서 에이전트입니다.

## 장기 메모리 사용 규칙

### 저장해야 할 정보 (→ /memories/ 경로)
- 사용자 선호도: `/memories/preferences.md`
- 작업 기록: `/memories/task_log.md`
- 중요 약속/일정: `/memories/schedule.md`
- 자주 쓰는 답변: `/memories/templates.md`

### 임시 저장 (→ 일반 경로)
- 현재 작업 중인 파일: `/workspace/`
- 계산 결과: `/tmp/`

### 시작 시 체크리스트
1. `/memories/` 폴더에서 사용자 관련 파일 확인
2. 이전 세션의 작업 기록 파악
3. 사용자 선호도에 맞게 응답 스타일 조정
"""

from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)

# 장기 메모리용 Store (실제로는 PostgresStore 권장)
memory_store = InMemoryStore()

# CompositeBackend 생성 (routes는 dict 형태)
composite_backend = CompositeBackend(
    default=StateBackend(),                          # 임시 저장
    routes={
        "/memories/": StoreBackend(store=memory_store),  # 영구 저장
    },
)

# 장기 메모리 에이전트 생성
longterm_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],
    system_prompt=longterm_memory_prompt,
    backend=composite_backend,  # CompositeBackend 연결
)

# 장기 메모리 에이전트 생성 완료
print(f"  /memories/ → StoreBackend (영구 저장)")
print(f"  그 외      → StateBackend (임시 저장)")


  /memories/ → StoreBackend (영구 저장)
  그 외      → StateBackend (임시 저장)


## 6. AsyncSubAgent — 비동기 서브에이전트

일반 서브에이전트는 메인 에이전트가 결과를 기다리는 **동기 방식**이에요. `AsyncSubAgent`를 사용하면 서브에이전트가 **백그라운드에서 실행**되고, 메인 에이전트는 다른 작업을 계속할 수 있어요.

```mermaid
flowchart TD
    MAIN["메인 에이전트"]
    
    subgraph SYNC_FLOW["동기 방식 (일반 서브에이전트)"]
        MAIN1["메인"] -->|"작업 위임"| SUB1["서브에이전트"]
        SUB1 -->|"대기 중..."| WAIT1["메인 대기 (블로킹)"]
        SUB1 -->|"완료 후 반환"| CONTINUE1["메인 재개"]
    end

    subgraph ASYNC_FLOW["비동기 방식 (AsyncSubAgent)"]
        MAIN2["메인"] -->|"백그라운드 시작"| ASYNC["AsyncSubAgent<br>(백그라운드 실행)"]
        MAIN2 -->|"즉시 계속"| OTHER["다른 작업 수행"]
        ASYNC -->|"전용 상태 채널 업데이트"| STATUS["진행 상황 스트리밍"]
        ASYNC -->|"완료 시 알림"| DONE["결과 통합"]
    end

    classDef main fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef sync fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef async_ fill:#d4edda,stroke:#28a745,color:#155724
    classDef status fill:#cce5ff,stroke:#007bff,color:#004085

    class MAIN1,MAIN2 main
    class SUB1,WAIT1,CONTINUE1 sync
    class ASYNC,OTHER,DONE async_
    class STATUS status
```

> 🔑 **핵심 개념**: `AsyncSubAgent`는 **전용 상태 채널**을 사용해요. 서브에이전트의 진행 상황을 `stream_mode="updates"`로 실시간 추적할 수 있어요. 오래 걸리는 작업(웹 크롤링, 대용량 데이터 처리 등)에 적합해요.

> 💡 **실무 팁**: UI를 제공하는 애플리케이션에서 `AsyncSubAgent`를 사용하면 "백그라운드에서 분석 중..." 같은 진행 상황 표시를 실시간으로 보여줄 수 있어요. 사용자 경험(UX)이 크게 향상돼요.

In [15]:
# ---------------------------------------------------
# AsyncSubAgent 개념 및 사용 패턴
# ---------------------------------------------------

# AsyncSubAgent 사용 패턴:
# ==================================================
async_code = '''
from deepagents import create_deep_agent, AsyncSubAgent
from langchain.messages import HumanMessage

# ---------------------------------------------------
# AsyncSubAgent 정의
# ---------------------------------------------------
# 오래 걸리는 작업에 적합해요 (웹 크롤링, 대용량 분석 등)
crawler_async = AsyncSubAgent(
    name="web_crawler",
    description=(
        "여러 웹 페이지를 순차적으로 크롤링하는 전문가입니다. "
        "오래 걸리는 작업이므로 백그라운드에서 실행해요."
    ),
    model="openai:gpt-4o-mini",
    # is_async=True: 백그라운드 실행, 전용 상태 채널 사용
)

# 메인 에이전트에 AsyncSubAgent 등록
main_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    subagents=[crawler_async],  # AsyncSubAgent 사용
    system_prompt=(
        "크롤링 작업을 web_crawler에게 위임하세요. "
        "크롤링이 진행되는 동안 다른 작업을 계속 수행할 수 있어요."
    ),
)
'''
print(async_code)

# ---
# 비동기 실행 및 스트리밍:
stream_code = '''
# stream_mode="updates"로 비동기 서브에이전트 상태 추적
for chunk in main_agent.stream(
    {"messages": [HumanMessage(content="5개 웹 페이지를 크롤링해줘")]},
    stream_mode="updates",
):
    for node_name, node_output in chunk.items():
        # 메인 에이전트 업데이트
        if node_name == "agent":
            print(f"[메인] {node_output}")
        # 비동기 서브에이전트 상태 채널 업데이트
        elif node_name.startswith("web_crawler"):
            print(f"[백그라운드] {node_output}")  # 진행 상황 실시간 출력
'''
print(stream_code)


from deepagents import create_deep_agent, AsyncSubAgent
from langchain.messages import HumanMessage

# ---------------------------------------------------
# AsyncSubAgent 정의
# ---------------------------------------------------
# 오래 걸리는 작업에 적합해요 (웹 크롤링, 대용량 분석 등)
crawler_async = AsyncSubAgent(
    name="web_crawler",
    description=(
        "여러 웹 페이지를 순차적으로 크롤링하는 전문가입니다. "
        "오래 걸리는 작업이므로 백그라운드에서 실행해요."
    ),
    model="openai:gpt-4o-mini",
    # is_async=True: 백그라운드 실행, 전용 상태 채널 사용
)

# 메인 에이전트에 AsyncSubAgent 등록
main_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    subagents=[crawler_async],  # AsyncSubAgent 사용
    system_prompt=(
        "크롤링 작업을 web_crawler에게 위임하세요. "
        "크롤링이 진행되는 동안 다른 작업을 계속 수행할 수 있어요."
    ),
)


# stream_mode="updates"로 비동기 서브에이전트 상태 추적
for chunk in main_agent.stream(
    {"messages": [HumanMessage(content="5개 웹 페이지를 크롤링해줘")]},
    stream_mode="updates",
):
    for node_name, node_output in chunk.items():
        # 메인 에이전트 업데이트
    

## 7. 실습: 컨텍스트 타입별 에이전트 설계하기

5가지 컨텍스트 타입을 종합해서 완전한 에이전트를 설계해봐요.

In [16]:
# ============================================================
# TODO: 5가지 컨텍스트를 모두 활용하는 에이전트 완성하기
#
# 목표: 사용자별 선호도를 기억하고, 긴 작업을 안전하게 수행하는 에이전트
#
# 힌트:
#   1. system_prompt에 역할, 규칙, /memories/ 경로 사용 지침 추가 (입력 컨텍스트)
#   2. config에 {"context": {"user_id": ..., "language": ...}} 추가 (런타임 컨텍스트)
#   3. 대용량 데이터 처리 시 파일에 저장 후 경로 참조 권고 (압축 컨텍스트)
#   4. 전문 작업은 subagents에 위임 (격리 컨텍스트)
#   5. CompositeBackend로 /memories/ → StoreBackend 설정 (장기 메모리)
#
# 예상 결과:
#   - 에이전트가 사용자 정보를 /memories/에 저장해요
#   - 다음 실행 시 이전 세션의 정보를 기억해요
#   - 런타임 컨텍스트로 사용자별 맞춤 응답을 제공해요
# ============================================================

from langgraph.store.memory import InMemoryStore

# TODO 1: 완성된 system_prompt 작성
complete_system_prompt = """
당신은 개인 학습 코치 에이전트입니다.

## 역할
# TODO: 역할을 추가해보세요

## 장기 메모리 규칙
# TODO: /memories/ 경로 사용 규칙을 추가해보세요
"""

# TODO 2: 런타임 컨텍스트 설정
user_runtime_config = {
    "configurable": {"thread_id": "coaching-session-001"},
    "context": {
        # TODO: user_id, language, goals 등을 추가해보세요
    }
}

# TODO 3: 격리 컨텍스트용 서브에이전트 정의
quiz_maker_subagent = {
    "name": "quiz_maker",
    "description": (
        # TODO: 퀴즈 생성 전문가 서브에이전트 설명을 작성해보세요
        "퀴즈를 만드는 전문가입니다."
    ),
    "system_prompt": "당신은 학습 퀴즈 전문가입니다. 주어진 주제로 퀴즈를 만들어요.",
    "model": "openai:gpt-4o-mini",
}

# TODO 4: CompositeBackend로 장기 메모리 설정
memory_store = InMemoryStore()

from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)

# TODO: composite_backend를 생성해보세요
# /memories/ → StoreBackend, 나머지 → StateBackend
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # TODO: /memories/ 경로 라우팅을 추가하세요
        "/memories/": StoreBackend(store=memory_store),
    },
)

# TODO 5: 5가지 컨텍스트를 모두 활용하는 에이전트 완성
complete_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],
    system_prompt=complete_system_prompt,         # 1. 입력 컨텍스트
    subagents=[quiz_maker_subagent],              # 4. 격리 컨텍스트
    backend=composite_backend,                    # 5. 장기 메모리
)

# 완성된 에이전트:
#   - 입력 컨텍스트: system_prompt 포함
#   - 런타임 컨텍스트: user_runtime_config에 정의
#   - 격리 컨텍스트: quiz_maker 서브에이전트
#   - 장기 메모리: /memories/ → StoreBackend
# 2. 런타임 컨텍스트는 invoke/stream 호출 시 config로 전달:
# complete_agent.invoke({...}, config=user_runtime_config)


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **입력 컨텍스트 (Input)**: `system_prompt`(역할/규칙), `AGENTS.md`(에이전트 자체 메모리), `SKILL.md`(Progressive Disclosure 스킬), 도구 docstring(tool prompts) 4가지 소스로 에이전트의 기본 지식을 설정해요.

- **런타임 컨텍스트 (Runtime)**: `config["context"]` 딕셔너리로 실행 시 동적 정보를 주입해요. 서브에이전트까지 자동 전파되어 멀티에이전트 시스템의 사용자 정보 공유가 간단해져요.

- **압축 컨텍스트 (Compression)**: 오프로딩(20K+ 토큰 데이터를 파일로 저장 후 경로 참조)과 요약(85% 임계값, LLM 자동 요약)으로 무한한 길이의 작업을 지원해요. 중요 정보는 항상 파일이나 Store에 저장해요.

- **격리 컨텍스트 (Isolation)**: 서브에이전트는 메인의 긴 히스토리 없이 fresh context에서 실행돼요. 컨텍스트 오염 방지와 토큰 비용 절감 두 가지 효과가 있어요.

- **장기 메모리 (Long-term)**: `CompositeBackend(StateBackend + StoreBackend)`를 사용해요. `/memories/` 경로는 `StoreBackend`로 라우팅해서 세션 간에 데이터를 유지해요. system_prompt에 경로 규칙을 명시하면 에이전트가 스스로 장기 메모리를 관리해요.

- **AsyncSubAgent**: 백그라운드에서 독립 실행되는 비동기 서브에이전트예요. 전용 상태 채널로 진행 상황을 실시간 스트리밍할 수 있어요.

## 다음 노트북 예고

다음 `04-Subagents.ipynb`에서는 **서브에이전트 심화 학습**을 해요. SubAgent 딕셔너리 vs CompiledSubAgent, 컨텍스트 전파 방식, 비동기 서브에이전트를 더 깊이 다뤄요.